# 05 — 5-fold cross-validation training

Trains one preset on every fold (each fold = one model trained on the other four folds). Folds that are already
finished — e.g. fold 0 from notebook 04 — are reused, not retrained. Outputs:

* one checkpoint and one set of predictions per fold (`outputs/checkpoints/<run>/`, `outputs/preds/<run>/`)
* **out-of-fold (OOF)** predictions: every training image is predicted by the model that did not train on it,
  so OOF accuracy uses all 63,660 images as validation
* a submission that averages the test probabilities of the fold models (no TTA; TTA and ensembling are in notebook 07)

This is the longest notebook (≈ 2 h per fold for ConvNeXt-Tiny at 128 px on the M1 Pro); run it in the background
from a terminal, see README.md.

In [ ]:
RUN = "convnext_tiny_shift"   # preset name
FOLDS = "0,1,2,3,4"           # folds to train
FORCE = False
MAKE_SUBMISSION = True

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))  # the notebooks live in the repo root, next to src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

import torch
from sklearn.metrics import confusion_matrix

from src.config import CLASSES, EXPERIMENTS_CSV, PRESETS, SUBMISSIONS_CSV, get_device, make_config, make_dirs
from src.data import load_test_df
from src.engine import fit, get_preds
from src.models import count_params, create_model
from src.plots import plot_confusion, plot_history, savefig
from src.utils import accuracy, softmax, write_submission

make_dirs()
print("device:", get_device())

from src.utils import log_loss

folds = [int(f) for f in FOLDS.split(",")]
cfg = make_config(RUN)
cfg

## 1. Training, fold by fold

In [ ]:
histories = {}
for fold in folds:
    histories[fold] = fit(cfg, fold=fold, force=FORCE)

## 2. Per-fold results and learning curves

In [ ]:
ex = pd.read_csv(EXPERIMENTS_CSV)
ex = ex[(ex["run"] == RUN) & ex["fold"].isin(folds)].sort_values("fold")
display(ex[["fold", "best_epoch", "epochs_run", "train_acc", "val_clean_acc", "val_shift_acc",
            "val_clean_loss", "val_shift_loss", "minutes"]])
display(ex[["val_clean_acc", "val_shift_acc"]].agg(["mean", "std", "min", "max"]).round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for fold, hist in histories.items():
    axes[0].plot(hist["epoch"], hist["train_loss"], marker=".", label=f"fold {fold}")
    axes[1].plot(hist["epoch"], hist["val_clean_acc"], marker=".", label=f"fold {fold}")
    axes[2].plot(hist["epoch"], hist["val_shift_acc"], marker=".", label=f"fold {fold}")
for ax, t in zip(axes, ["train loss", "val_clean accuracy", "val_shift accuracy"]):
    ax.set_title(t)
    ax.set_xlabel("epoch")
    ax.legend(fontsize=8)
plt.suptitle(RUN)
plt.tight_layout()
savefig(f"05_folds_{RUN}")
plt.show()

> **Observations (to be written by me):**

## 3. Out-of-fold predictions

In [ ]:
oof = {}
for split in ["val_clean", "val_shift"]:
    parts = [get_preds(RUN, f, split) for f in folds]
    oof[split] = dict(ids=np.concatenate([p[0] for p in parts]), probs=softmax(np.concatenate([p[1] for p in parts])),
                      labels=np.concatenate([p[2] for p in parts]))
summary = pd.DataFrame({split: {"images": len(o["labels"]), "accuracy": accuracy(o["probs"], o["labels"]),
                                "log loss": log_loss(o["probs"], o["labels"])} for split, o in oof.items()})
display(summary.round(4))

per_class = pd.DataFrame({split: [(o["probs"].argmax(1)[o["labels"] == k] == k).mean() for k in range(len(CLASSES))]
                          for split, o in oof.items()}, index=CLASSES)
display(per_class.round(4))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (split, o) in zip(axes, oof.items()):
    plot_confusion(confusion_matrix(o["labels"], o["probs"].argmax(1), labels=range(len(CLASSES))), CLASSES,
                   title=f"{RUN} OOF {split}", ax=ax)
plt.tight_layout()
savefig(f"05_oof_confusion_{RUN}")
plt.show()

> **Observations (to be written by me):**

## 4. Submission: average of the fold models' test probabilities (no TTA)

In [ ]:
if MAKE_SUBMISSION:
    test_df = load_test_df()
    probs = np.mean([softmax(get_preds(RUN, f, "test")[1]) for f in folds], axis=0)
    agree = np.mean([(softmax(get_preds(RUN, f, "test")[1]).argmax(1) == probs.argmax(1)).mean() for f in folds])
    print(f"mean agreement of single fold models with the fold average on test: {agree:.4f}")
    write_submission(test_df, probs, f"{RUN}_cv{len(folds)}", runs=RUN,
                     val_clean=round(summary.loc["accuracy", "val_clean"], 4),
                     val_shift=round(summary.loc["accuracy", "val_shift"], 4),
                     notes=f"mean of folds {FOLDS}, no TTA (val = OOF)")
    display(pd.read_csv(SUBMISSIONS_CSV))